# Full Pipeline for Live Sign Language Translation with Citizen ASL Dataset

This notebook implements a complete pipeline for converting sign language video into live text predictions using pose estimation and a language model.

## Pipeline Overview
1. **Landmark Extraction**: Extract body and hand landmarks from Citizen ASL sign videos using [MediaPipe Holistic](https://developers.google.com/mediapipe).
2. **Preprocessing** : Normalize and split the landmark sequences into training, validation, and test sets.
3. **Model Training** : Train a deep learning model on the preprocessed data.
4. **Live translation** : Use a trained model and LLM in a real-time translation system.

## Requirements:
- Install dependencies via:
    ```bash
    pip install -r requirements.txt 
    ```
    **Or**, install them manually. Make sure you are using the following versions:
    - Python: 3.10.17
    - TensorFlow: 2.19.0
    - MediaPipe 0.10.9
    - OpenAI 1.82.0

- Obtain an API key for a Large Language Model (LLM):
    - This pipeline uses [OpenAI's GPT API](https://platform.openai.com/) for language enhancement in the live translation phase.
    - Note: OpenAI is not free. You must have a valid and funded API key.

- **Landmark extraction** requires [ASL Citizen Dataset](https://www.microsoft.com/en-us/research/project/asl-citizen/) downloaded

> **Preprocessed data** and **model training** is available on [this notebook](https://www.kaggle.com/code/tobypu/aslcitizen-top200-training).

> **Live translation** is available for testing using a trained model covering 200 unique glosses (271 total classes including duplicates). To use it, specify the path to the trained model and the accompanying index_to_glos_200.json file, which maps model outputs to gloss labels.

In [2]:
import sys
import tensorflow as tf
import mediapipe as mp
import openai
import cv2

print("Python:", sys.version)
print("TensorFlow:", tf.__version__)
print("MediaPipe:", mp.__version__)
print("OpenAI:", openai.__version__)
print("cv2:", cv2.__version__)


Python: 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
TensorFlow: 2.19.0
MediaPipe: 0.10.9
OpenAI: 1.82.0
cv2: 4.11.0


The following modules are imported for different stages of the pipeline:

In [3]:
from utils import asl_citizen_MP_encoding
from utils import preprocessing_split 
from utils import train_model 
from utils import live_translation

## 1. Landmark Extraction
This step converts raw sign videos from the Citizen ASL dataset into landmark sequences using MediaPipe Holistic.

### Input
- Sign videos from the Citizen ASL dataset
- Split CSV files: `train.csv`, `test.csv`, and `val.csv` under `splits/` directory
- Citizen ASL dataset folders and videos should not be modified or renamed from their original structure.

### What It Does
For each video, the pipeline:
1. Extracts 3D landmarks for:
    - 33 pose landmarks
    - 21 left-hand landmarks
    - 21 right-hand landmarks
    - A subset of 32 facial landmarks
2. Combines them into `(frames, 107, 3)` NumPy array
3. Stores only the`(x,y)` coordinates and `gloss` to a `.npz` file.
### Output
- One `.npz` file per video saved under `processed_save_dir`
- Each `.npz` file contains:
    - `landmarks`: `np.darray` of shape (frames,107,2)
    - `gloss`: The sign label


> **Note** : Approximately takes about 3-4 days to process all videos with Mediapipe Holistic.

In [4]:
video_dir='/ASL_Citizen/videos'
processed_save_dir='processed_all'
asl_citizen_MP_encoding.process_asl_dataset(video_dir,processed_save_dir)

FileNotFoundError: [Errno 2] No such file or directory: '/ASL_Citizen\\splits\\train.csv'

## 2. Preprocessing
This step prepared the extracted landmarks for model training by normalizing, pad the data, and encode the labels.

### What It Does
1. Generate Label and One-Hot Encoders
    - Reads the glosses from `train.csv`, `val.csv`, or `test.csv`
    - Creates:
        - A label encoder (gloss -> integer)
        - A one-hot encoder (integer -> one-hot vector)
        - A gloss_to_index.json dictionary for mapping (gloss -> integer)
2. Encode Labels
    - For each `.npz` file:
        - Loads the gloss label
        - Converts it to one-hot format using the encoders
    - Saves the resulting arrays as `y_train.npy`, `y_val.npy`, and `y_test.npy` under `save_dir`
3. Preprocess Landmark Features
    - Loads landmark arrays of shape `(frames, 107, 2)`
    - Removes:
        - Pose landmarks 0-10 and 23-32
    - Pads or repeats frame to a fixed length of 150
    - Applies normalization:
        - Anchor-based normalization for body, face, and arm landmarks.
        - Hand normalization to bound hand keypoints in `[-0.5,0.5]`, centered at `(0,0)`
    - Saves the processed data as `x_train.npy`, `x_val.npy`, and `x_test.npy` in shape `(videos, 160,86 , 2)` under `save_dir`

### Input
- `.npz` files from the landmark extrtaction step (under `save_dir_name/{split}`)
- Corresponding `train.csv`, `val.csv`, or `test.csv` file

### Output
- One-hot encoded label arrays: `y_train.npy`, `y_val.npy`, and `y_test.npy`
- Preprocessed landmark arrays: `x_train.npy`, `x_val.npy`, and `x_test.npy`

> Note: This step applies pose normalization, arm alignment, and hand bounding box to remove spatial and body proportionality bias. For a detailed explanation and justification of these choices, please refer  to [our research paper](https://doi.org/10.26877/sj5scb03)

In [5]:
# === 1. Generate Label Encoder, OneHot Encoder, and Gloss Dictionary ===
csv_path='ASL_Citizen/splits/test.csv' #location of test/train/val.csv
save_dict_path='gloss_to_index' #Path and dictionary name
label_encoder, onehot_encoder, gloss_to_index = preprocessing_split.generate_gloss_dictionary(csv_path,save_dict_path)

FileNotFoundError: [Errno 2] No such file or directory: 'ASL_Citizen/splits/test.csv'

In [6]:
len(gloss_to_index) #STRING (GlOSS) TO UNIQUE INTEGER (0-2730)

NameError: name 'gloss_to_index' is not defined

In [ ]:
save_dir = 'model_train_data'
processed_save_dir='/processed_all'

#Create y_train, y_test, and y_val
preprocessing_split.encode_labels(label_encoder, onehot_encoder, processed_save_dir, save_dir)

#Create X_train, X_test, and X_val
preprocessing_split.preprocess_and_save_x(processed_save_dir, save_dir, split='train')
preprocessing_split.preprocess_and_save_x(processed_save_dir, save_dir, split='test')
preprocessing_split.preprocess_and_save_x(processed_save_dir, save_dir, split='val')

### 3. Model Training

This section trains a GRU-based model using the preprocessed sign language data. The training logic is defined in `train_model.py`, which consists of two main components:

#### A. `data_loader(...)`
This function loads and optionally filters the training, validation, and test datasets:
- Loads `.npy` files containing pose features and one-hot encoded labels.
- Due to prediction results, labels are converted back from one-hot encoded vectors into unique integer class labels.
- Supports partial gloss filtering by mapping similar gloss variants (ex: `GO1` and `GO2` -> `GO`). 
- Optionally merges the test set into training for cases like leaderboard training.
- Returns: tuples of `(X_train, y_train)`, `(X_val, y_val)`, `(X_test, y_test)`, and a decoder dictionary (if filtered) (decoder maps integer -> gloss).

### B. `train_gru_model(...)`
Trains a sequential GRU-based neural network using TensorFlow/Keras:
- Two GRU layers (`386` units and followed by `192` units).
- Dropout layers for regularization.
- Final `Dense` layer with `softmax` for multi-class classification.
- Automatically saves the **best-performing model** on validation accuracy.
- If test data is passed, accuracy and macro F1 score are computed.
- The final model is saved as `best_model.keras`.

---

Training was performed on **Kaggle** using GPU acceleartion. The notebook is available [here](https://www.kaggle.com/code/tobypu/aslcitizen-top200-training)





We selected the **top 200 most commonly used signs** based on [HandSpeak's list of most-used signs](https://www.handspeak.com/word/most-used/). These signs are listed in `.txt`file, each separtated by a newline. 

We also moved test data into train for better interference.

In [ ]:
(train_X, train_y), (val_X, val_y), (_, _), decoder = train_model.data_loader(
    data_dir="/kaggle/input/citizen-asl-mediapipe-encoded-and-preprocessed",
    gloss_to_index_dir="/kaggle/input/citizen-asl-mediapipe-encoded-and-preprocessed/gloss_to_index.json",
    filtered_txt_path="/kaggle/input/top200citizen/Citizen200.txt",
    merge_test_to_train=True
)

print("Training classes:", len(set(train_y)))
print("Validation classes:", len(set(val_y)))
print("Decoder entries:", len(decoder))
print("Training label range:", min(train_y), "to", max(train_y))
print("Decoder first 5:", list(decoder.items())[:5])
print("Decoder last 5:", list(decoder.items())[-5:])


FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/citizen-asl-mediapipe-encoded-and-preprocessed/x_train.npy'

: 

#### Model summary

In [4]:
from tensorflow.keras.models import load_model
model=load_model('models/best_model200.keras')
# Show a summary of the model architecture
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru (GRU)                       │ (None, 150, 386)       │       648,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 150, 386)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_1 (GRU)                     │ (None, 192)            │       334,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 192)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 271)            │        52,303 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,104,591 (11.84 MB)

 Trainable params: 1,034,863 (3.95 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2,069,728 (7.90 MB)

### 4. Live translation

The `start_live_feed()` function is responsible for capturing webcam input, performing real-time sign language recognition using a trained GRU model, and generating meaningful translations through a language model (OpenAI GPT).

#### Input Arguments
   - `model_path`: Path to the `.keras` model file containing the trained GRU network for recognizing sequences of body/keypoint features.
   - `encoder_path`: JSON file mapping predicted class indices to sign language glosses (e.g., `{0: "HELLO", 1: "THANK-YOU", ...}`).
   - `client`: Load OpenAI API key stored in `.env`
   - `threshold` : Controls the sensitivity to motion. A lower value makes the system more sensitive, triggering capture with smaller movements. A higher value requires more movement to start capturing, making it less sensitive. Default is `1.2` (float)
   - `webcam` : Specifies which webcam to use webcam device. Default is `0`. If you have multiple, try `1`, `2`, etc (integer)
   - `complexity_setting` : Sets the model complexity for the MediaPipe Holistic pipeline. (`0` : Fastest but least accurate, `1` : Balanced, `2` : Most accurate but slowest). Default is `0`

#### **Flowchart Summary**
![Flowchart](flowchart.jpg)

#### **Creating `.env` API KEY file**
1. Open a text editor and add your OpenAI API key like this:
    ```bash
    API_KEY=sk-xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx
    ```
2. Save the file as `.env` in your project repository

In [6]:
from utils import live_translation

In [7]:
from utils import live_translation

print("Import successful")
print("Module location:", live_translation.__file__)

Import successful
Module location: c:\Users\Aryanth Venugopal\live-asl-translation\utils\live_translation.py


In [ ]:
live_translation.start_live_feed(
    model_path='models/best_model200_new.keras',
    encoder_path='models/index_to_gloss_200.json',
    client=None
)


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 586ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step


In [5]:
import inspect
from utils import live_translation

print(inspect.getsource(live_translation.start_live_feed))

def start_live_feed(
        model_path,
        encoder_path,
        client,
        threshold=1.2,
        webcam=0,
        complexity_setting=0
        ):

    ##################################################
    # == Define local functions for feed translation #
    ##################################################
    def movement_score(landmarks_frames,hand_weight=3):
        '''
        Calculate the average movement between consecutive frames.

        Parameters:
        landmark_frames (numpy array):  Sequence of landmarks with frames with shape (T,N,2), 
                                        where T is number of frames, N is the number of landmarks,
                                        and 2 corresponds to x and coordinates.

        hand_weight (integer)       :   An integer scaling factor for hands landmark (>0). A higher
                                        value increases the contribution of hand movement towards the final score

        Output:
        floa

In [1]:
import json

with open("models/index_to_gloss_200.json", "r") as f:
    glosses = json.load(f)

print("Number of labels:", len(glosses))

for index, gloss in glosses.items():
    print(index, ":", gloss)

Number of labels: 209
0 : ABOUT1
1 : ABOUT2
2 : AFTER
3 : AGAIN
4 : ALL
5 : ANSWER
6 : ANY
7 : ARRIVE
8 : ASK
9 : BAD
10 : BECAUSE
11 : BEFORE
12 : BETTER
13 : BIG
14 : BOY
15 : BRING
16 : BUSY
17 : BUT
18 : BUY
19 : CAN1
20 : CAN2
21 : CHANGE
22 : CLEAN
23 : CLOSE
24 : COLD
25 : COLOR
26 : COME
27 : DAY
28 : DEAF1
29 : DEAF2
30 : DIFFERENT
31 : DRINK1
32 : DRINK2
33 : DRIVE
34 : EARLY
35 : EASY
36 : EAT1
37 : EAT2
38 : EMAIL
39 : ENOUGH
40 : EXCUSE
41 : FAMILY
42 : FAR
43 : FAST
44 : FEEL
45 : FEW
46 : FIND
47 : FINE1
48 : FINE2
49 : FINGERSPELL
50 : FINISH
51 : FOLLOW1
52 : FOLLOW2
53 : FOLLOW3
54 : FOLLOW4
55 : FOR
56 : FRIEND
57 : FROM
58 : GET
59 : GIRL
60 : GIVE
61 : GO
62 : GONE
63 : GOOD
64 : HAPPEN
65 : HAPPY
66 : HARD
67 : HAVE
68 : HE
69 : HEARING
70 : HELLO
71 : HELP
72 : HERE
73 : HOME
74 : HOT
75 : HOW1
76 : HOW2
77 : HURT
78 : IMPROVE
79 : IN
80 : KEEP
81 : KNOW
82 : LAST
83 : LATE
84 : LATER
85 : LEARN
86 : LEAVE
87 : LIKE
88 : LIVE1
89 : LIVE2
90 : LOOKFOR
91 : MAKE
92

In [ ]:
import inspect
import utils.live_translation as lt

print(inspect.getsource(lt.handle_capture_logic))

In [3]:
print("Notebook is working!")

Notebook is working!


In [4]:
import inspect
import utils.live_translation as lt

print(inspect.getsource(lt.start_live_feed))

def start_live_feed(
        model_path,
        encoder_path,
        client,
        threshold=1.2,
        webcam=0,
        complexity_setting=0
        ):

    ##################################################
    # == Define local functions for feed translation #
    ##################################################
    def movement_score(landmarks_frames,hand_weight=3):
        '''
        Calculate the average movement between consecutive frames.

        Parameters:
        landmark_frames (numpy array):  Sequence of landmarks with frames with shape (T,N,2), 
                                        where T is number of frames, N is the number of landmarks,
                                        and 2 corresponds to x and coordinates.

        hand_weight (integer)       :   An integer scaling factor for hands landmark (>0). A higher
                                        value increases the contribution of hand movement towards the final score

        Output:
        floa

In [5]:
import json
import numpy as np
from tensorflow.keras.models import load_model

model = load_model("models/best_model200.keras")

with open("models/index_to_gloss_200.json", "r") as f:
    labels = json.load(f)

print("Model output shape:", model.output_shape)
print("Number of labels:", len(labels))
print("First label:", labels.get("0"))
print("Last label:", labels.get(str(len(labels) - 1)))

Model output shape: (None, 271)
Number of labels: 209
First label: ABOUT1
Last label: YOUR


In [6]:
import json

with open("models/index_to_gloss_200.json", "r") as f:
    labels = json.load(f)

print("Number of labels:", len(labels))
print("Highest label index:", max(map(int, labels.keys())))
print("Missing indices:", [
    i for i in range(max(map(int, labels.keys())) + 1)
    if str(i) not in labels
])

Number of labels: 209
Highest label index: 208
Missing indices: []


In [7]:
with open("utils/asl_citizen_MP_encoding.py", "r", encoding="utf-8") as f:
    print(f.read())

import pandas as pd
import numpy as np
import os
from tqdm import tqdm
import cv2
import mediapipe as mp

#Define Python Functions

# Functions receives video path as an input, and numpy array (frames, 107 landmarks, 3) as an output 
def extract_landmarks_from_video(video_path):
    important_face_landmarks = [
        33, 133, 159, 145, 153, 144,          # Right eye
        362, 263, 386, 374, 380, 373,         # Left eye
        70, 63, 105, 66, 107,                 # Right eyebrow
        295, 282, 320, 285, 318,              # Left eyebrow
        1, 168, 197, 4,                       # Nose bridge and tip
        78, 308, 13, 14, 81, 311              # Mouth and lips
        ]
    mp_holistic = mp.solutions.holistic
    cap = cv2.VideoCapture(video_path)
    all_landmarks = [] #Temporary list to store landmarks extracted from each frame of the video.
    with mp_holistic.Holistic(static_image_mode=False,  #Mediapipe configuration parameters
                              model_com

In [8]:
import json
from tensorflow.keras.models import load_model

model = load_model("models/best_model200.keras")

with open("models/index_to_gloss_200.json", "r") as f:
    decoder = json.load(f)

print("Model output classes:", model.output_shape[-1])
print("Decoder entries:", len(decoder))
print("Decoder index range:", min(map(int, decoder)), "to", max(map(int, decoder)))
print("First 10 decoder entries:", list(decoder.items())[:10])

Model output classes: 271
Decoder entries: 209
Decoder index range: 0 to 208
First 10 decoder entries: [('0', 'ABOUT1'), ('1', 'ABOUT2'), ('2', 'AFTER'), ('3', 'AGAIN'), ('4', 'ALL'), ('5', 'ANSWER'), ('6', 'ANY'), ('7', 'ARRIVE'), ('8', 'ASK'), ('9', 'BAD')]


In [1]:
import json

with open("models/index_to_gloss.json", "r") as f:
    labels = json.load(f)

print("Number of labels:", len(labels))
print("First 10 entries:", list(labels.items())[:10])
print("Last 10 entries:", list(labels.items())[-10:])
print("Highest index:", max(map(int, labels.keys())))

FileNotFoundError: [Errno 2] No such file or directory: 'models/index_to_gloss.json'

In [2]:
import os

print("Current folder:", os.getcwd())
print("\nFiles inside models:")
print(os.listdir("models"))

Current folder: C:\Users\Aryanth Venugopal\live-asl-translation

Files inside models:
['best_model200.keras', 'index_to_gloss_200.json']


In [3]:
import os

for root, dirs, files in os.walk("."):
    dirs[:] = [d for d in dirs if d not in [".venv", ".git", "__pycache__"]]
    for file in files:
        if file.endswith(".json"):
            print(os.path.join(root, file))

.\models\index_to_gloss_200.json


In [5]:
import utils.train_model as train_model

In [6]:
import inspect
print(inspect.getsource(train_model.data_loader))

def data_loader(data_dir,
                gloss_to_index_dir, 
                filtered_txt_path=None, 
                merge_test_to_train=False
                ):
    X_train = np.load(f"{data_dir}/x_train.npy")
    y_train = np.load(f"{data_dir}/y_train.npy")

    X_test = np.load(f"{data_dir}/x_test.npy")
    y_test = np.load(f"{data_dir}/y_test.npy")

    X_val = np.load(f"{data_dir}/x_val.npy")
    y_val = np.load(f"{data_dir}/y_val.npy")

    X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], -1)
    X_test = X_test.reshape(X_test.shape[0], X_test.shape[1], -1)
    X_val = X_val.reshape(X_val.shape[0], X_val.shape[1], -1)

    y_train = np.argmax(y_train, axis=1)
    y_val   = np.argmax(y_val, axis=1)
    y_test  = np.argmax(y_test, axis=1)

    decoder = None

    #If user uses partial gloss instead all the unique gloss from ASL Citizen
    if filtered_txt_path:
        #Load original gloss_to_index dictionary
        with open(gloss_to_index_dir) as f:
            g

In [1]:
print("Training samples:", len(train_X))
print("Training labels:", len(train_y))
print("Unique training classes:", len(set(train_y)))
print("Minimum class index:", min(train_y))
print("Maximum class index:", max(train_y))

NameError: name 'train_X' is not defined

In [2]:
from tensorflow.keras.models import load_model

model = load_model("models/best_model200.keras")

print("Input shape:", model.input_shape)
print("Output shape:", model.output_shape)
print("Final layer:", model.layers[-1].get_config())

Input shape: (None, 150, 172)
Output shape: (None, 271)
Final layer: {'name': 'dense', 'trainable': True, 'dtype': {'module': 'keras', 'class_name': 'DTypePolicy', 'config': {'name': 'float32'}, 'registered_name': None}, 'units': 271, 'activation': 'softmax', 'use_bias': True, 'kernel_initializer': {'module': 'keras.initializers', 'class_name': 'GlorotUniform', 'config': {'seed': None, 'input_axes': None, 'output_axes': None}, 'registered_name': None}, 'bias_initializer': {'module': 'keras.initializers', 'class_name': 'Zeros', 'config': {}, 'registered_name': None}, 'kernel_regularizer': None, 'bias_regularizer': None, 'kernel_constraint': None, 'bias_constraint': None, 'quantization_config': None}


In [3]:
import inspect
import utils.live_translation as lt

print(inspect.getsource(lt.start_live_feed))

def start_live_feed(
        model_path,
        encoder_path,
        client,
        threshold=1.2,
        webcam=0,
        complexity_setting=0
        ):

    ##################################################
    # == Define local functions for feed translation #
    ##################################################
    def movement_score(landmarks_frames,hand_weight=3):
        '''
        Calculate the average movement between consecutive frames.

        Parameters:
        landmark_frames (numpy array):  Sequence of landmarks with frames with shape (T,N,2), 
                                        where T is number of frames, N is the number of landmarks,
                                        and 2 corresponds to x and coordinates.

        hand_weight (integer)       :   An integer scaling factor for hands landmark (>0). A higher
                                        value increases the contribution of hand movement towards the final score

        Output:
        floa

In [4]:
import inspect
import utils.preprocessing_split as preprocessing

print(inspect.getsource(preprocessing.pad_video))

def pad_video(X):
    padded_length = 150
    data_array=np.zeros((padded_length,X.shape[1],X.shape[2]))
    for landmark in range(0,X.shape[1]):
        for coordinate in range(0,X.shape[2]):
            data_array[:,landmark,coordinate] = np.tile(X[:,landmark,coordinate],int(padded_length/X.shape[0]+2))[:padded_length]
    return data_array



In [5]:
import numpy as np
from utils.preprocessing_split import pad_video

# Simulate 30 frames, 107 landmarks, 2 coordinates
test_data = np.zeros((30, 107, 2), dtype=np.float32)

padded = pad_video(test_data)
model_input = padded.reshape(padded.shape[0], -1)
model_input = np.expand_dims(model_input, axis=0)

print("Padded shape:", padded.shape)
print("Model input shape:", model_input.shape)
print("Expected model input:", model.input_shape)

Padded shape: (150, 107, 2)
Model input shape: (1, 150, 214)
Expected model input: (None, 150, 172)


In [6]:
import inspect
import utils.live_translation as lt

print(inspect.getsource(lt.extract_landmarks))
print(inspect.getsource(lt.preprocess))

def extract_landmarks(results):
    """
    Extracts pose (filtered), left/right hands, and selected face landmarks from MediaPipe results,
    returning only the x and y coordinates.
    
    Parameters:
        results (mp_holistic.HolisticResults): The MediaPipe Holistic results object.
        important_face_landmarks (list): Indices of important face landmarks to extract.
    
    Returns:
        np.ndarray: Combined landmark array with shape (N, 2), where N is the total number of selected keypoints.
    """
    important_face_landmarks = [
        33, 133, 159, 145, 153, 144,          # Right eye
        362, 263, 386, 374, 380, 373,         # Left eye
        70, 63, 105, 66, 107,                 # Right eyebrow
        295, 282, 320, 285, 318,              # Left eyebrow
        1, 168, 197, 4,                       # Nose bridge and tip
        78, 308, 13, 14, 81, 311              # Mouth and lips
        ]
    
    # Indices to remove from pose landmarks
    indices_to_remo

In [7]:
import json

with open("models/index_to_gloss_200.json", "r") as f:
    labels = json.load(f)

print("Labels:", len(labels))
print("Highest label index:", max(map(int, labels.keys())))
print("Sample labels:", list(labels.items())[:20])

Labels: 209
Highest label index: 208
Sample labels: [('0', 'ABOUT1'), ('1', 'ABOUT2'), ('2', 'AFTER'), ('3', 'AGAIN'), ('4', 'ALL'), ('5', 'ANSWER'), ('6', 'ANY'), ('7', 'ARRIVE'), ('8', 'ASK'), ('9', 'BAD'), ('10', 'BECAUSE'), ('11', 'BEFORE'), ('12', 'BETTER'), ('13', 'BIG'), ('14', 'BOY'), ('15', 'BRING'), ('16', 'BUSY'), ('17', 'BUT'), ('18', 'BUY'), ('19', 'CAN1')]


In [8]:
from tensorflow.keras.models import load_model

model = load_model("models/best_model200.keras")
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ gru (GRU)                            │ (None, 150, 386)            │         648,480 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 150, 386)            │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ gru_1 (GRU)                          │ (None, 192)                 │         334,080 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 192)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 271)                 │          52,303 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 3,104,591 (11.84 MB)

 Trainable params: 1,034,863 (3.95 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2,069,728 (7.90 MB)

In [1]:
import tensorflow as tf

model = tf.keras.models.load_model(
    "models/best_model200.keras",
    compile=False
)

print("Model input shape:", model.input_shape)
print("Model output shape:", model.output_shape)
print("Output classes:", model.output_shape[-1])

Model input shape: (None, 150, 172)
Model output shape: (None, 271)
Output classes: 271


In [2]:

import os
import json

print("Project files:")
for root, dirs, files in os.walk("models"):
    for file in files:
        print(os.path.join(root, file))

print("\nNotebook files:")
for file in os.listdir("."):
    if file.endswith((".ipynb", ".json", ".txt", ".keras")):
        print(file)

Project files:
models\best_model200.keras
models\index_to_gloss_200.json

Notebook files:
best_model200_original.keras
gloss_to_index_unfiltered.json
main_livefeed.ipynb
requirements.txt


In [3]:
import tensorflow as tf

for path in [
    "models/best_model200.keras",
    "best_model200_original.keras"
]:
    model = tf.keras.models.load_model(path, compile=False)
    print(path)
    print("  Input:", model.input_shape)
    print("  Output:", model.output_shape)

models/best_model200.keras
  Input: (None, 150, 172)
  Output: (None, 271)
best_model200_original.keras
  Input: (None, 150, 172)
  Output: (None, 271)


In [4]:
import hashlib

for path in [
    "models/best_model200.keras",
    "best_model200_original.keras"
]:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    print(path, h.hexdigest())

models/best_model200.keras 589f40d6522f6b2df58caf385bcb24f526f555db1eec3a71c7d2a7b09f8ac1f8
best_model200_original.keras 589f40d6522f6b2df58caf385bcb24f526f555db1eec3a71c7d2a7b09f8ac1f8
